# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process a dataset defined by a Croissant schema using the `mlcroissant` library.

### Dataset Source
The dataset's FAIR² Croissant schema is available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load dataset metadata and records with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their associated `@id`s.

In [ ]:
# Get record sets and overview their fields
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - Field name: {field.name}, @id: {field.id}, dataType: {getattr(field, 'dataType', None)}")
    print()

### Example: Inspect the first record from each record set
`mlcroissant.Dataset.records(record_set=...)` yields structured record dicts for each overall record.


In [ ]:
for rs in record_sets:
    print(f"--- RecordSet: {rs.name} (@id: {rs.id}) ---")
    records_iter = dataset.records(record_set=rs.id)
    try:
        first = next(records_iter)
        pprint.pprint(first)
    except StopIteration:
        print("No records found.")
    print()

## 3. Data Extraction
Load full data from each available record set into a DataFrame. All accesses use the record set `@id` and field `@id`s.

In [ ]:
# Prepare DataFrames for each record set
dataframes = {}
# Gather all record set @ids
record_set_ids = [r.id for r in record_sets]
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for {record_set_id} with {len(df)} records. Columns:")
        print(list(df.columns))
        print()
    else:
        print(f"No records found for {record_set_id}.")

### Show sample records
We'll display the head of the main record set (the one containing most tabular data).

In [ ]:
# Select primary (largest) record set for analysis
if dataframes:
    # Pick the first non-empty table
    main_record_set_id = max(dataframes, key=lambda k: len(dataframes[k]))
    display_id = main_record_set_id
    print(f"Previewing records from record set '@id': {display_id}\n")
    display_cols = dataframes[display_id].columns.tolist()
    print("Columns (@id):", display_cols)
    # Show 5 records
    display(dataframes[display_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Let's filter, transform, and group data using only field `@id` references.

> **All code operates using field and record set `@id`s only.**

In [ ]:
# Identify a numeric field by scanning the fields in our main record set
main_recordset = None
for rs in record_sets:
    if rs.id == display_id:
        main_recordset = rs
        break

numeric_field_id = None
for field in main_recordset.fields:
    # Select a field with dataType == Float or Integer
    dtype = getattr(field, 'dataType', None)
    if dtype and (dtype.endswith('Float') or dtype.endswith('Integer') or dtype.endswith('Number')):
        numeric_field_id = field.id
        print(f"Using numeric field: {field.name} (@id: {numeric_field_id}, dataType: {dtype})")
        break

# For demonstration, if we can't deduce, try an int/float column
if numeric_field_id is None:
    # Fallback: pick first int/float column from dataframe
    for col in dataframes[display_id].columns:
        if pd.api.types.is_numeric_dtype(dataframes[display_id][col]):
            numeric_field_id = col
            print(f"Fallback numeric field: {numeric_field_id}")
            break

# Set a numerical threshold for filtering
threshold = 10

if numeric_field_id and numeric_field_id in dataframes[display_id]:
    # Remove outliers/records below this value
    filtered_df = dataframes[display_id][dataframes[display_id][numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' (z-score):")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Group by another field if any categorical field is available
    group_field_id = None
    for field in main_recordset.fields:
        # Exclude numeric
        dtype = getattr(field, 'dataType', None)
        if dtype and dtype.endswith('Text') and field.id in filtered_df.columns:
            group_field_id = field.id
            print(f"Grouping by categorical field: {field.name} (@id: {group_field_id})")
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean of {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df)
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize value distributions and relationships using only field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in filtered_df:
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(8,4))
        sns.boxplot(
            data=filtered_df,
            x=group_field_id, y=numeric_field_id
        )
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

We demonstrated how to parse and analyze a richly structured clinical dataset using the `mlcroissant` library, referencing record sets and fields solely by their `@id` as per the Croissant schema. This framework supports robust and reproducible data handling and analysis. You may now proceed with deeper analysis or modeling using these programmatic building blocks.
